# Spark SQL and DataFrames TP

## Imports and Initialization

In [1]:
from pyspark.sql import SparkSession

In [2]:
spark = SparkSession.builder \
    .appName("Spark SQL TP") \
    .getOrCreate()

sc = spark.sparkContext

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/26 10:15:56 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/04/26 10:15:56 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/04/26 10:15:56 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.


## Creating Dataframes

### Common ways
    
    1. From an RDD
    2. From a data source (csv, json, etc)
    3. From a table
    4. From an SQL statement

#### From an RDD

In [3]:
first_rdd = sc.parallelize((
  (1, "Batman"),
  (2, "Superman"),
  (3, "Spiderman")
))

first_df = spark.createDataFrame(first_rdd)

#### Inspecting the schema

In [4]:
# let's see how the dataframe and its schema look like

# first_df.show(5)
# first_df.schema  # schema object
first_df.printSchema()  # prints schema of dataframe

root
 |-- _1: long (nullable = true)
 |-- _2: string (nullable = true)



#### Creating a DataFrame from a data source / file

In [7]:
df = (spark.read
      .option("sep", "\t")  # separator
      .option("header", False)  #  file has no header row
      .option("inferSchema", True)  # spark tries to infer data types
      .csv("data/movie_data.tsv")
)

df.printSchema()

root
 |-- _c0: integer (nullable = true)
 |-- _c1: integer (nullable = true)
 |-- _c2: integer (nullable = true)
 |-- _c3: integer (nullable = true)



#### Creating/Changing a header
    First column - User ID
    Second column - movieID
    Third column - User rating (between 1-5)
    Fourth column - Timestamp

In [8]:
movies_df = df.toDF("userID", "movieID", "rating", "timestamp")
movies_df.printSchema()

root
 |-- userID: integer (nullable = true)
 |-- movieID: integer (nullable = true)
 |-- rating: integer (nullable = true)
 |-- timestamp: integer (nullable = true)



In [9]:
movies_df.explain("simple")
# movies_df.explain("extended")
# movies_df.explain("formatted")

== Physical Plan ==
*(1) Project [_c0#21 AS userID#29, _c1#22 AS movieID#30, _c2#23 AS rating#31, _c3#24 AS timestamp#32]
+- FileScan csv [_c0#21,_c1#22,_c2#23,_c3#24] Batched: false, DataFilters: [], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/opt/workspace/data/movie_data.tsv], PartitionFilters: [], PushedFilters: [], ReadSchema: struct<_c0:int,_c1:int,_c2:int,_c3:int>




In [10]:
movies_df.select("movieID", "rating").take(5)

[Row(movieID=242, rating=3),
 Row(movieID=302, rating=3),
 Row(movieID=377, rating=1),
 Row(movieID=51, rating=2),
 Row(movieID=346, rating=1)]

In [11]:
# movie IDs with rating >= 4
movies_df.select(movies_df.movieID).where("rating >= 4").distinct().count()

1447

In [12]:
from pyspark.sql.functions import col

movies_df.select(col("movieID")).take(5)

[Row(movieID=242),
 Row(movieID=302),
 Row(movieID=377),
 Row(movieID=51),
 Row(movieID=346)]

## Writing DataFrames

    1. Into tables/views
    2. Into files

### Storing DataFrames in Tables
The table will be accessible from other clusters and it will be persisted.

In [13]:
(movies_df.write
 .mode("overwrite")  # ignore, append
 .saveAsTable("movie_rating")
)

spark.sql("SELECT COUNT(*) FROM movie_rating").show()

+--------+
|count(1)|
+--------+
|  100000|
+--------+



It is also possible to create a temporary, session-based view.

In [14]:
movies_df.createOrReplaceTempView("temp_movie_rating")

spark.sql("SELECT COUNT(*) FROM temp_movie_rating").show()

+--------+
|count(1)|
+--------+
|  100000|
+--------+



In [15]:
# read back a whole table into a DataFrame
movie_tables_df = spark.table("movie_rating")

display(movie_tables_df)

DataFrame[userID: int, movieID: int, rating: int, timestamp: int]

### Creating DataFrames with SQL

In [16]:
movie_sql_df = spark.sql("SELECT movieID, rating, timestamp as cdate FROM movie_rating LIMIT 50")

display(movie_sql_df)
movie_sql_df.count()

DataFrame[movieID: int, rating: int, cdate: int]

50

### Using Parquet and Modes

In [17]:
(first_df.write
  .option("compression", "snappy")
  .mode("overwrite")  # overwrites data if exists. other options "append", "error", "ignore"
  .parquet("/tmp/dataframe-training/first.parquet") # path
)

# test after running overwrite, append, ignore modes:
display(spark.read.parquet("/tmp/dataframe-training/first.parquet"))
spark.read.parquet("/tmp/dataframe-training/first.parquet").count()

DataFrame[_1: bigint, _2: string]

3

## DataFrame Methods

### Column Transformations

#### Creating column objects

In [18]:
from pyspark.sql.functions import col

movies_df.select(col("userID")).take(5)
# userID = movies_df.userID
# userID = movies_df["userID"]
# movies_df.select(userID).take(5)

[Row(userID=196),
 Row(userID=186),
 Row(userID=22),
 Row(userID=244),
 Row(userID=166)]

#### Selecting columns

In [19]:
userID_df = movies_df.select("userID", "rating")
# userID_df = movies_df.select(col("userID"), col("rating"))
# userID_df = movies_df.select(col("userID"), col("rating"))
# userID_df = movies_df.select(col("*"))  # selecting all values - SQL syntax

# Fields can be nested
# userID_df = movies_df.select(col("userID"), col("movieID"), col("timestamp.hour"))  # selecting nested fields (struct) with dot notation
# if the time stamp was a nested object with format hour, minute and seconds

display(userID_df)
userID_df.take(2)

DataFrame[userID: int, rating: int]

[Row(userID=196, rating=3), Row(userID=186, rating=3)]

#### Adding and Replacing Columns

In [20]:
high_rating_df = movies_df.withColumn("high_rating", col("rating").isin(4, 5))

display(high_rating_df)
high_rating_df.take(15)

DataFrame[userID: int, movieID: int, rating: int, timestamp: int, high_rating: boolean]

[Row(userID=196, movieID=242, rating=3, timestamp=881250949, high_rating=False),
 Row(userID=186, movieID=302, rating=3, timestamp=891717742, high_rating=False),
 Row(userID=22, movieID=377, rating=1, timestamp=878887116, high_rating=False),
 Row(userID=244, movieID=51, rating=2, timestamp=880606923, high_rating=False),
 Row(userID=166, movieID=346, rating=1, timestamp=886397596, high_rating=False),
 Row(userID=298, movieID=474, rating=4, timestamp=884182806, high_rating=True),
 Row(userID=115, movieID=265, rating=2, timestamp=881171488, high_rating=False),
 Row(userID=253, movieID=465, rating=5, timestamp=891628467, high_rating=True),
 Row(userID=305, movieID=451, rating=3, timestamp=886324817, high_rating=False),
 Row(userID=6, movieID=86, rating=3, timestamp=883603013, high_rating=False),
 Row(userID=62, movieID=257, rating=2, timestamp=879372434, high_rating=False),
 Row(userID=286, movieID=1014, rating=5, timestamp=879781125, high_rating=True),
 Row(userID=200, movieID=222, rating

#### Renaming Columns

In [21]:
time_df = movies_df.withColumnRenamed("timestamp", "time")
display(time_df)
display(movies_df)

DataFrame[userID: int, movieID: int, rating: int, time: int]

DataFrame[userID: int, movieID: int, rating: int, timestamp: int]

#### Replacing Column Values

In [22]:
from pyspark.sql.functions import when

high_rating = movies_df.select(
    col("*"),
    when(col("rating") == 5, 10)
    .when(col("rating") == 4, 8)
    .when(col("rating") == 3, 6)
    .alias("inflated_rating")
)
display(high_rating)
high_rating.show()

DataFrame[userID: int, movieID: int, rating: int, timestamp: int, inflated_rating: int]

+------+-------+------+---------+---------------+
|userID|movieID|rating|timestamp|inflated_rating|
+------+-------+------+---------+---------------+
|   196|    242|     3|881250949|              6|
|   186|    302|     3|891717742|              6|
|    22|    377|     1|878887116|           NULL|
|   244|     51|     2|880606923|           NULL|
|   166|    346|     1|886397596|           NULL|
|   298|    474|     4|884182806|              8|
|   115|    265|     2|881171488|           NULL|
|   253|    465|     5|891628467|             10|
|   305|    451|     3|886324817|              6|
|     6|     86|     3|883603013|              6|
|    62|    257|     2|879372434|           NULL|
|   286|   1014|     5|879781125|             10|
|   200|    222|     5|876042340|             10|
|   210|     40|     3|891035994|              6|
|   224|     29|     3|888104457|              6|
|   303|    785|     3|879485318|              6|
|   122|    387|     5|879270459|             10|


### Row Transformations
#### Filtering DataFrames

In [23]:
low_rating = movies_df.filter("rating > 4")  # with RDDs: filter(lambda row: row["rating"] > 4)

low_rating.take(5)

[Row(userID=253, movieID=465, rating=5, timestamp=891628467),
 Row(userID=286, movieID=1014, rating=5, timestamp=879781125),
 Row(userID=200, movieID=222, rating=5, timestamp=876042340),
 Row(userID=122, movieID=387, rating=5, timestamp=879270459),
 Row(userID=38, movieID=95, rating=5, timestamp=892430094)]

In [24]:
low_rating.count()

21201

In [25]:
movies_df.count()

100000

#### Deduplicating Rows

In [26]:
distinct_rows_df = movies_df.distinct()
distinct_rows_df.count()

# no_duplicates = spark.sql("SELECT DISTINCT(movieID) FROM movie_rating")
# no_duplicates.count()

100000

In [27]:
distinct_movies = movies_df.select("movieID").distinct()
distinct_movies.count()

1682

#### Limiting

In [28]:
limit_df = movies_df.limit(25)
display(limit_df)
limit_df.count()

DataFrame[userID: int, movieID: int, rating: int, timestamp: int]

25

#### Sorting

In [29]:
increasing_ts_df = movies_df.sort("timestamp")
increasing_ts_df.take(5)

decreasing_ts_df = movies_df.sort(col("timestamp").desc())
decreasing_ts_df.take(5)
# display(decreasing_ts_df)

[Row(userID=729, movieID=333, rating=4, timestamp=893286638),
 Row(userID=729, movieID=748, rating=4, timestamp=893286638),
 Row(userID=729, movieID=689, rating=4, timestamp=893286638),
 Row(userID=729, movieID=328, rating=3, timestamp=893286638),
 Row(userID=729, movieID=313, rating=3, timestamp=893286638)]

In [30]:
# orderBy is same as sort

ordered_df = movies_df.orderBy(["movieID", "rating"])  # same as .sort(["movieID", "rating"])
ordered_df.take(5)

[Row(userID=199, movieID=1, rating=1, timestamp=883782854),
 Row(userID=609, movieID=1, rating=1, timestamp=886896185),
 Row(userID=761, movieID=1, rating=1, timestamp=876190094),
 Row(userID=424, movieID=1, rating=1, timestamp=880859493),
 Row(userID=15, movieID=1, rating=1, timestamp=879455635)]

### Aggregations

In [31]:
# creating grouped objects

movies_grp = movies_df.groupBy("movieID")
# user_grp = movies_df.groupBy("userID")

#### Counting

In [32]:
# group and count

movies_grp.count().show()
# user_grp.count().show()

+-------+-----+
|movieID|count|
+-------+-----+
|    496|  231|
|    471|  221|
|    463|   71|
|    148|  128|
|   1342|    2|
|    833|   49|
|   1088|   13|
|   1591|    6|
|   1238|    8|
|   1580|    1|
|   1645|    1|
|    392|   68|
|    623|   39|
|    540|   43|
|    858|    3|
|    737|   59|
|    243|  132|
|   1025|   44|
|   1084|   21|
|   1127|   11|
+-------+-----+
only showing top 20 rows



#### Average

In [33]:
movies_df.groupBy("movieID").avg("rating").show()

+-------+------------------+
|movieID|       avg(rating)|
+-------+------------------+
|    496| 4.121212121212121|
|    471|3.6108597285067874|
|    463| 3.859154929577465|
|    148|          3.203125|
|   1342|               2.5|
|    833| 3.204081632653061|
|   1088| 2.230769230769231|
|   1591|3.1666666666666665|
|   1238|             3.125|
|   1580|               1.0|
|   1645|               4.0|
|    392|3.5441176470588234|
|    623| 2.923076923076923|
|    540| 2.511627906976744|
|    858|               1.0|
|    737| 2.983050847457627|
|    243|2.4393939393939394|
|   1025|2.9318181818181817|
|   1084| 3.857142857142857|
|   1127| 2.909090909090909|
+-------+------------------+
only showing top 20 rows



#### Sum

In [34]:
movies_df.groupBy("movieID").sum("rating").show()

+-------+-----------+
|movieID|sum(rating)|
+-------+-----------+
|    496|        952|
|    471|        798|
|    463|        274|
|    148|        410|
|   1342|          5|
|    833|        157|
|   1088|         29|
|   1591|         19|
|   1238|         25|
|   1580|          1|
|   1645|          4|
|    392|        241|
|    623|        114|
|    540|        108|
|    858|          3|
|    737|        176|
|    243|        322|
|   1025|        129|
|   1084|         81|
|   1127|         32|
+-------+-----------+
only showing top 20 rows



#### More Complex Aggregations
- Use `agg()` for applying different types of aggregations
- Also allows for other transformations on top of the returned column

In [35]:
from pyspark.sql.functions import sum, avg, approx_count_distinct

movies_df.groupBy("movieID").agg(avg("rating").alias("total_purchases")).show()

+-------+------------------+
|movieID|   total_purchases|
+-------+------------------+
|    496| 4.121212121212121|
|    471|3.6108597285067874|
|    463| 3.859154929577465|
|    148|          3.203125|
|   1342|               2.5|
|    833| 3.204081632653061|
|   1088| 2.230769230769231|
|   1591|3.1666666666666665|
|   1238|             3.125|
|   1580|               1.0|
|   1645|               4.0|
|    392|3.5441176470588234|
|    623| 2.923076923076923|
|    540| 2.511627906976744|
|    858|               1.0|
|    737| 2.983050847457627|
|    243|2.4393939393939394|
|   1025|2.9318181818181817|
|   1084| 3.857142857142857|
|   1127| 2.909090909090909|
+-------+------------------+
only showing top 20 rows



### Datetime Functions

#### Casting to DateTime

In [36]:
from pyspark.sql.functions import to_date, from_unixtime, to_timestamp

timestamp_df = movies_df.withColumn("datetime", to_timestamp(from_unixtime(col("timestamp"))))

display(timestamp_df)
timestamp_df.show(5)

DataFrame[userID: int, movieID: int, rating: int, timestamp: int, datetime: timestamp]

+------+-------+------+---------+-------------------+
|userID|movieID|rating|timestamp|           datetime|
+------+-------+------+---------+-------------------+
|   196|    242|     3|881250949|1997-12-04 15:55:49|
|   186|    302|     3|891717742|1998-04-04 19:22:22|
|    22|    377|     1|878887116|1997-11-07 07:18:36|
|   244|     51|     2|880606923|1997-11-27 05:02:03|
|   166|    346|     1|886397596|1998-02-02 05:33:16|
+------+-------+------+---------+-------------------+
only showing top 5 rows



#### Formatting Dates

In [37]:
from pyspark.sql.functions import date_format

format_df = (timestamp_df.withColumn("date string", date_format("datetime", "MMMM dd, yyyy"))
  .withColumn("time string", date_format("datetime", "HH:mm:ss.SSSSSS"))
) 
display(format_df)
format_df.show(5)

DataFrame[userID: int, movieID: int, rating: int, timestamp: int, datetime: timestamp, date string: string, time string: string]

+------+-------+------+---------+-------------------+-----------------+---------------+
|userID|movieID|rating|timestamp|           datetime|      date string|    time string|
+------+-------+------+---------+-------------------+-----------------+---------------+
|   196|    242|     3|881250949|1997-12-04 15:55:49|December 04, 1997|15:55:49.000000|
|   186|    302|     3|891717742|1998-04-04 19:22:22|   April 04, 1998|19:22:22.000000|
|    22|    377|     1|878887116|1997-11-07 07:18:36|November 07, 1997|07:18:36.000000|
|   244|     51|     2|880606923|1997-11-27 05:02:03|November 27, 1997|05:02:03.000000|
|   166|    346|     1|886397596|1998-02-02 05:33:16|February 02, 1998|05:33:16.000000|
+------+-------+------+---------+-------------------+-----------------+---------------+
only showing top 5 rows



In [38]:
format_df.explain("extended")

== Parsed Logical Plan ==
'Project [userID#29, movieID#30, rating#31, timestamp#32, datetime#333, date string#364, date_format('datetime, HH:mm:ss.SSSSSS, None) AS time string#371]
+- Project [userID#29, movieID#30, rating#31, timestamp#32, datetime#333, date_format(datetime#333, MMMM dd, yyyy, Some(Etc/UTC)) AS date string#364]
   +- Project [userID#29, movieID#30, rating#31, timestamp#32, to_timestamp(from_unixtime(cast(timestamp#32 as bigint), yyyy-MM-dd HH:mm:ss, Some(Etc/UTC)), None, TimestampType, Some(Etc/UTC), false) AS datetime#333]
      +- Project [_c0#21 AS userID#29, _c1#22 AS movieID#30, _c2#23 AS rating#31, _c3#24 AS timestamp#32]
         +- Relation [_c0#21,_c1#22,_c2#23,_c3#24] csv

== Analyzed Logical Plan ==
userID: int, movieID: int, rating: int, timestamp: int, datetime: timestamp, date string: string, time string: string
Project [userID#29, movieID#30, rating#31, timestamp#32, datetime#333, date string#364, date_format(datetime#333, HH:mm:ss.SSSSSS, Some(Etc/UTC)

#### Extracting date parts

In [39]:
from pyspark.sql.functions import year, month, dayofweek, minute, second

dt_col = col("datetime")

datetime_df = (timestamp_df
    .withColumn("year", year(dt_col))
    .withColumn("month", month(dt_col))
    .withColumn("dayofweek", dayofweek(dt_col))
    .withColumn("minute", minute(dt_col))
    .withColumn("second", second(dt_col))              
)

display(datetime_df)
datetime_df.show(5)

DataFrame[userID: int, movieID: int, rating: int, timestamp: int, datetime: timestamp, year: int, month: int, dayofweek: int, minute: int, second: int]

+------+-------+------+---------+-------------------+----+-----+---------+------+------+
|userID|movieID|rating|timestamp|           datetime|year|month|dayofweek|minute|second|
+------+-------+------+---------+-------------------+----+-----+---------+------+------+
|   196|    242|     3|881250949|1997-12-04 15:55:49|1997|   12|        5|    55|    49|
|   186|    302|     3|891717742|1998-04-04 19:22:22|1998|    4|        7|    22|    22|
|    22|    377|     1|878887116|1997-11-07 07:18:36|1997|   11|        6|    18|    36|
|   244|     51|     2|880606923|1997-11-27 05:02:03|1997|   11|        5|     2|     3|
|   166|    346|     1|886397596|1998-02-02 05:33:16|1998|    2|        2|    33|    16|
+------+-------+------+---------+-------------------+----+-----+---------+------+------+
only showing top 5 rows



#### Converting to date

In [40]:
from pyspark.sql.functions import to_date

date_df = timestamp_df.withColumn("date", to_date(col("datetime")))
display(date_df)
date_df.show(5)

DataFrame[userID: int, movieID: int, rating: int, timestamp: int, datetime: timestamp, date: date]

+------+-------+------+---------+-------------------+----------+
|userID|movieID|rating|timestamp|           datetime|      date|
+------+-------+------+---------+-------------------+----------+
|   196|    242|     3|881250949|1997-12-04 15:55:49|1997-12-04|
|   186|    302|     3|891717742|1998-04-04 19:22:22|1998-04-04|
|    22|    377|     1|878887116|1997-11-07 07:18:36|1997-11-07|
|   244|     51|     2|880606923|1997-11-27 05:02:03|1997-11-27|
|   166|    346|     1|886397596|1998-02-02 05:33:16|1998-02-02|
+------+-------+------+---------+-------------------+----------+
only showing top 5 rows



#### Manipulating dates

In [41]:
from pyspark.sql.functions import date_add

plus_df = timestamp_df.withColumn("plus_two_days", date_add(col("datetime"), 2))

# plus_df = timestamp_df.selectExpr("*","datetime + interval 2 days")  # spark sql allows for +/- interval type of datetime manipulation

display(plus_df)
plus_df.show(5)

DataFrame[userID: int, movieID: int, rating: int, timestamp: int, datetime: timestamp, plus_two_days: date]

+------+-------+------+---------+-------------------+-------------+
|userID|movieID|rating|timestamp|           datetime|plus_two_days|
+------+-------+------+---------+-------------------+-------------+
|   196|    242|     3|881250949|1997-12-04 15:55:49|   1997-12-06|
|   186|    302|     3|891717742|1998-04-04 19:22:22|   1998-04-06|
|    22|    377|     1|878887116|1997-11-07 07:18:36|   1997-11-09|
|   244|     51|     2|880606923|1997-11-27 05:02:03|   1997-11-29|
|   166|    346|     1|886397596|1998-02-02 05:33:16|   1998-02-04|
+------+-------+------+---------+-------------------+-------------+
only showing top 5 rows



## User Defined Functions (UDFs)

### Importing a New Dataset

In [43]:
countries_df = spark.read.option("header", True).option("inferSchema", True).csv("data/geo.csv")
# countries_df = spark.read.csv("../data/geo.csv", header=True, inferSchema=True)

display(countries_df)
countries_df.show(5)

DataFrame[country: string, latitude: double, longitude: double, name: string]

+-------+---------+----------+--------------------+
|country| latitude| longitude|                name|
+-------+---------+----------+--------------------+
|     AD|42.546245|  1.601554|             Andorra|
|     AE|23.424076| 53.847818|United Arab Emirates|
|     AF| 33.93911| 67.709953|         Afghanistan|
|     AG|17.060816|-61.796428| Antigua and Barbuda|
|     AI|18.220554|-63.068615|            Anguilla|
+-------+---------+----------+--------------------+
only showing top 5 rows



### Calculating the Distance between two Latitude/Longitude Pairs
Source: https://gist.github.com/rochacbruno/2883505

In [44]:
import math

from pyspark.sql.types import DoubleType
from pyspark.sql.functions import udf

def distance(startLat, startLon, endLat, endLon):    

    radius = 6371  # in km
    dlat = math.radians(endLat - startLat)
    dlon = math.radians(endLon - startLon)
    a = math.sin(dlat/2) * math.sin(dlat/2) + math.cos(math.radians(startLat)) \
        * math.cos(math.radians(endLat)) * math.sin(dlon/2) * math.sin(dlon/2)
    c = 2 * math.atan2(math.sqrt(a), math.sqrt(1-a))
    d = radius * c

    return d

geo_distance = udf(distance, DoubleType()) # you can define the return type, default is string
geo_distance

<function __main__.distance(startLat, startLon, endLat, endLon)>

In [45]:
# Let's compare distances of each country's geographical centre, using our UDF

distance_df = (countries_df
  # .na.drop()
  # .filter(col("country") == "EE")
 .join(countries_df  # no join keys = cartesian/cross join
       # .filter(col("country") == "FI")
       .toDF("join_country","join_latitude","join_longitude","join_name")  # we can use .toDF() to rename all columns of a dataframe
       # .na.drop() # remove countries with null values
      )
 .withColumn("distance_in_km", geo_distance("latitude","longitude","join_latitude","join_longitude"))  # here we call the UDF
  
 # .filter(col("distance_in_km").cast(DoubleType()) > 0)
 # .orderBy("distance_in_km")
)

display(distance_df)
distance_df.show()

DataFrame[country: string, latitude: double, longitude: double, name: string, join_country: string, join_latitude: double, join_longitude: double, join_name: string, distance_in_km: double]

+-------+---------+---------+-------+------------+-------------+--------------+--------------------+------------------+
|country| latitude|longitude|   name|join_country|join_latitude|join_longitude|           join_name|    distance_in_km|
+-------+---------+---------+-------+------------+-------------+--------------+--------------------+------------------+
|     AD|42.546245| 1.601554|Andorra|          AD|    42.546245|      1.601554|             Andorra|               0.0|
|     AD|42.546245| 1.601554|Andorra|          AE|    23.424076|     53.847818|United Arab Emirates| 5219.961592950525|
|     AD|42.546245| 1.601554|Andorra|          AF|     33.93911|     67.709953|         Afghanistan| 5705.717586470977|
|     AD|42.546245| 1.601554|Andorra|          AG|    17.060816|    -61.796428| Antigua and Barbuda| 6569.940071966785|
|     AD|42.546245| 1.601554|Andorra|          AI|    18.220554|    -63.068615|            Anguilla|6591.8235986165355|
|     AD|42.546245| 1.601554|Andorra|   

## Complex Types
 
We will use **another dataset** containing information about songs on Spotify.

For more data and a more detailed hands on, see https://github.com/anindya-saha/Data-Science-with-Spark/blob/master/working-with-nested-data-types/working-with-nested-data-types.ipynb

In [47]:
spotify_df = spark.read.csv(path='data/spotify-songs.csv', inferSchema=True, header=True)

In [48]:
spotify_df.printSchema()

root
 |-- id: integer (nullable = true)
 |-- song_title: string (nullable = true)
 |-- artist: string (nullable = true)
 |-- acousticness: double (nullable = true)
 |-- danceability: double (nullable = true)
 |-- duration_ms: integer (nullable = true)
 |-- energy: double (nullable = true)
 |-- instrumentalness: double (nullable = true)
 |-- key: integer (nullable = true)
 |-- liveness: double (nullable = true)
 |-- loudness: double (nullable = true)
 |-- mode: integer (nullable = true)
 |-- speechiness: double (nullable = true)
 |-- tempo: double (nullable = true)
 |-- time_signature: integer (nullable = true)
 |-- valence: double (nullable = true)
 |-- target: integer (nullable = true)



In [49]:
spotify_df.show(10)

+---+-----------------+----------------+------------+------------+-----------+------+----------------+---+--------+--------+----+-----------+-------+--------------+-------+------+
| id|       song_title|          artist|acousticness|danceability|duration_ms|energy|instrumentalness|key|liveness|loudness|mode|speechiness|  tempo|time_signature|valence|target|
+---+-----------------+----------------+------------+------------+-----------+------+----------------+---+--------+--------+----+-----------+-------+--------------+-------+------+
|  0|         Mask Off|          Future|      0.0102|       0.833|     204600| 0.434|          0.0219|  2|   0.165|  -8.795|   1|      0.431|150.062|             4|  0.286|     1|
|  1|          Redbone|Childish Gambino|       0.199|       0.743|     326933| 0.359|         0.00611|  1|   0.137| -10.401|   1|     0.0794|160.083|             4|  0.588|     1|
|  2|     Xanny Family|          Future|      0.0344|       0.838|     185707| 0.412|         2.34E-

### Nested Types

Our goal is to

1. Combine the columns `'key', 'mode', 'target'` into an **array**.
   - An array contains a list of elements of _the same type_.
2. Transform the acoustic qualities
   ```
   'acousticness', 'tempo', 'liveness', 'instrumentalness', 'energy', 'danceability', 'speechiness', 'loudness'
   ```
    of a song from individual columns into a **map**.
    - A map contains key-value types.
    - All keys have to be of the same type, as do values.

In general, the keys for a map can be obtained from another column, but here we use the `lit` function to define constants (also called literals) instead.


In [50]:
from pyspark.sql.functions import array, create_map, lit

spotify_map_df = (spotify_df.select(
    'id',
    'song_title',
    'artist',
    'duration_ms',
    array('key', 'mode', 'target').alias('audience'),
    create_map(
        lit('acousticness'), 'acousticness', 
        lit('danceability'), 'acousticness',
        lit('energy'), 'energy',
        lit('instrumentalness'), 'instrumentalness',
        lit('liveness'), 'liveness',
        lit('loudness'), 'loudness',
        lit('speechiness'), 'speechiness',
        lit('tempo'), 'tempo'
    ).alias('qualities'),
    'time_signature',
    'valence',
)  # .cache()   # Example use of cache
)

In [51]:
spotify_map_df.printSchema()

root
 |-- id: integer (nullable = true)
 |-- song_title: string (nullable = true)
 |-- artist: string (nullable = true)
 |-- duration_ms: integer (nullable = true)
 |-- audience: array (nullable = false)
 |    |-- element: integer (containsNull = true)
 |-- qualities: map (nullable = false)
 |    |-- key: string
 |    |-- value: double (valueContainsNull = true)
 |-- time_signature: integer (nullable = true)
 |-- valence: double (nullable = true)



In [52]:
spotify_map_df.show(10, truncate=True)    # try with truncate=True (default value) or without truncate option

+---+-----------------+----------------+-----------+----------+--------------------+--------------+-------+
| id|       song_title|          artist|duration_ms|  audience|           qualities|time_signature|valence|
+---+-----------------+----------------+-----------+----------+--------------------+--------------+-------+
|  0|         Mask Off|          Future|     204600| [2, 1, 1]|{acousticness -> ...|             4|  0.286|
|  1|          Redbone|Childish Gambino|     326933| [1, 1, 1]|{acousticness -> ...|             4|  0.588|
|  2|     Xanny Family|          Future|     185707| [2, 1, 1]|{acousticness -> ...|             4|  0.173|
|  3|   Master Of None|     Beach House|     199413| [5, 1, 1]|{acousticness -> ...|             4|   0.23|
|  4|   Parallel Lines|     Junior Boys|     392893| [5, 0, 1]|{acousticness -> ...|             4|  0.904|
|  5|         Sneakin’|           Drake|     251333| [8, 1, 1]|{acousticness -> ...|             4|  0.264|
|  6|      Childs Play|     

#### Write the new DataFrame to a JSON file

In [53]:
spotify_map_df.coalesce(1).write.json(path='spotify-songs-json', mode="overwrite")

The `coalesce(1)` function reduces the number of partitions of the DataFrame to 1.

Since Spark creates one file for each partition, by using `coalesce(1)`, you're asking Spark to write entire DataFrame into a single JSON file.

Note: Using `coalesce(1)` is typically done when you want a single output file, which is common in cases where you need a compact, single-file format for easier consumption or when a system expects a single file as output. However, this can hurt performance when working with large datasets, as it forces Spark to consolidate everything into a single partition.

#### Reload the restructured DataFrame using a more complex schema with Nested Data Types

In [54]:
from pyspark.sql.types import StructType, StructField, ArrayType, MapType, StringType, IntegerType

nested_schema = StructType((
    StructField('id', IntegerType(), nullable=False),
    StructField('song_title', StringType(), nullable=False),
    StructField('artist', StringType(), nullable=False),
    StructField('duration_ms', IntegerType(), nullable=False),
    StructField('audience',
                ArrayType(elementType=IntegerType()),
                nullable=False
    ),
    StructField('qualities',
                MapType(keyType=StringType(), valueType=DoubleType(), valueContainsNull=False),
                nullable=True
    ),
    StructField('time_signature', IntegerType(), nullable=False),
    StructField('valence', DoubleType(), nullable=False),
))

In [55]:
spotify_reloaded_df = spark.read.json(path='spotify-songs-json', schema=nested_schema)

In [56]:
spotify_reloaded_df.show(10, truncate=False)

+---+-----------------+----------------+-----------+----------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------+--------------+-------+
|id |song_title       |artist          |duration_ms|audience  |qualities                                                                                                                                                                    |time_signature|valence|
+---+-----------------+----------------+-----------+----------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------+--------------+-------+
|0  |Mask Off         |Future          |204600     |[2, 1, 1] |{acousticness -> 0.0102, danceability -> 0.0102, energy -> 0.434, instrumentalness -> 0.0219, liveness -> 0.165, loudness -> -8.795, speechiness -> 0.431,

In [57]:
spotify_reloaded_df.printSchema()

root
 |-- id: integer (nullable = true)
 |-- song_title: string (nullable = true)
 |-- artist: string (nullable = true)
 |-- duration_ms: integer (nullable = true)
 |-- audience: array (nullable = true)
 |    |-- element: integer (containsNull = true)
 |-- qualities: map (nullable = true)
 |    |-- key: string
 |    |-- value: double (valueContainsNull = true)
 |-- time_signature: integer (nullable = true)
 |-- valence: double (nullable = true)



#### Extracting array elements
Recall that `audience` column is a combination of three attributes `key`, `mode`, and `target`.
We can extract each element into its own column again.

In [58]:
spotify_map_df.select(
    'song_title',
    col('audience').getItem(0).alias('key'), 
    col('audience').getItem(1).alias('mode'),
    col('audience').getItem(2).alias('target'),
).show(5)

+--------------+---+----+------+
|    song_title|key|mode|target|
+--------------+---+----+------+
|      Mask Off|  2|   1|     1|
|       Redbone|  1|   1|     1|
|  Xanny Family|  2|   1|     1|
|Master Of None|  5|   1|     1|
|Parallel Lines|  5|   0|     1|
+--------------+---+----+------+
only showing top 5 rows



#### Extracting map values
Recall that the `qualities` column is a map created from multiple attributes of a song.
We can extract (some of) these attributes again and store them in individual columns.

In [59]:
spotify_map_df.select(
    'song_title',
    col('qualities').getItem('acousticness').alias('acousticness'),
    col('qualities').getItem('speechiness').alias('speechiness'),
).limit(10).show()

+-----------------+------------+-----------+
|       song_title|acousticness|speechiness|
+-----------------+------------+-----------+
|         Mask Off|      0.0102|      0.431|
|          Redbone|       0.199|     0.0794|
|     Xanny Family|      0.0344|      0.289|
|   Master Of None|       0.604|     0.0261|
|   Parallel Lines|        0.18|     0.0694|
|         Sneakin’|     0.00479|      0.185|
|      Childs Play|      0.0145|      0.156|
|  Gyöngyhajú lány|      0.0202|     0.0371|
|I've Seen Footage|      0.0481|      0.347|
|   Digital Animal|     0.00208|      0.237|
+-----------------+------------+-----------+



#### A bit more condensed

In [60]:
# For a more concise and to generate a more efficient parsed logical plan.

cols = [col("song_title")] + list(map(
        lambda f: col("qualities").getItem(f).alias(str(f)), ["acousticness", "speechiness", "liveness", "tempo"]))

spotify_map_df.select(cols).show(10)

+-----------------+------------+-----------+--------+-------+
|       song_title|acousticness|speechiness|liveness|  tempo|
+-----------------+------------+-----------+--------+-------+
|         Mask Off|      0.0102|      0.431|   0.165|150.062|
|          Redbone|       0.199|     0.0794|   0.137|160.083|
|     Xanny Family|      0.0344|      0.289|   0.159| 75.044|
|   Master Of None|       0.604|     0.0261|  0.0922| 86.468|
|   Parallel Lines|        0.18|     0.0694|   0.439|174.004|
|         Sneakin’|     0.00479|      0.185|   0.164| 85.023|
|      Childs Play|      0.0145|      0.156|   0.207|  80.03|
|  Gyöngyhajú lány|      0.0202|     0.0371|    0.16|144.154|
|I've Seen Footage|      0.0481|      0.347|   0.342|130.035|
|   Digital Animal|     0.00208|      0.237|   0.571| 99.994|
+-----------------+------------+-----------+--------+-------+
only showing top 10 rows



#### Extracting map values programmatically
In five steps.

Manually appending the columns is fine if we know all the distinct keys in the map.
If we don’t know all the distinct keys, we’ll need a programatic solution, but be warned – this approach is slow!

**Step 1: Create a DataFrame with all the unique keys**

In [61]:
from pyspark.sql.functions import collect_set, map_keys, explode

keys_df = spotify_map_df.select(explode(map_keys(col("qualities")))).distinct()

In [62]:
keys_df.show(truncate=False)

+----------------+
|col             |
+----------------+
|acousticness    |
|danceability    |
|tempo           |
|energy          |
|liveness        |
|instrumentalness|
|speechiness     |
|loudness        |
+----------------+



In [63]:
keys_df.explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- HashAggregate(keys=[col#1071], functions=[])
   +- Exchange hashpartitioning(col#1071, 200), ENSURE_REQUIREMENTS, [plan_id=1368]
      +- HashAggregate(keys=[col#1071], functions=[])
         +- Generate explode(map_keys(qualities#838)), false, [col#1071]
            +- Project [map(acousticness, acousticness#719, danceability, acousticness#719, energy, energy#722, instrumentalness, instrumentalness#723, liveness, liveness#725, loudness, loudness#726, speechiness, speechiness#728, tempo, tempo#729) AS qualities#838]
               +- FileScan csv [acousticness#719,energy#722,instrumentalness#723,liveness#725,loudness#726,speechiness#728,tempo#729] Batched: false, DataFilters: [], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/opt/workspace/data/spotify-songs.csv], PartitionFilters: [], PushedFilters: [], ReadSchema: struct<acousticness:double,energy:double,instrumentalness:double,liveness:double,loudness:double,...




**Step 2: Convert the DataFrame to a list with all the unique keys**

In [64]:
keys = [row[0] for row in keys_df.collect()]
keys

['acousticness',
 'danceability',
 'tempo',
 'energy',
 'liveness',
 'instrumentalness',
 'speechiness',
 'loudness']

**Step 3: Create an array of column objects for the map items**

In [65]:
key_cols = list(map(lambda f: col("qualities").getItem(f).alias(str(f)), keys))

**Step 4: Add any additional columns before calculating the final result**

In [66]:
final_cols = [col("song_title")] + key_cols

**Step 5: Run a `select()` to get the final result**

In [67]:
spotify_map_df.select(final_cols).show(10)

+-----------------+------------+------------+-------+------+--------+----------------+-----------+--------+
|       song_title|acousticness|danceability|  tempo|energy|liveness|instrumentalness|speechiness|loudness|
+-----------------+------------+------------+-------+------+--------+----------------+-----------+--------+
|         Mask Off|      0.0102|      0.0102|150.062| 0.434|   0.165|          0.0219|      0.431|  -8.795|
|          Redbone|       0.199|       0.199|160.083| 0.359|   0.137|         0.00611|     0.0794| -10.401|
|     Xanny Family|      0.0344|      0.0344| 75.044| 0.412|   0.159|         2.34E-4|      0.289|  -7.148|
|   Master Of None|       0.604|       0.604| 86.468| 0.338|  0.0922|            0.51|     0.0261| -15.236|
|   Parallel Lines|        0.18|        0.18|174.004| 0.561|   0.439|           0.512|     0.0694| -11.648|
|         Sneakin’|     0.00479|     0.00479| 85.023|  0.56|   0.164|             0.0|      0.185|  -6.682|
|      Childs Play|      0.0

In [68]:
spotify_map_df.select(final_cols).explain(True)

== Parsed Logical Plan ==
'Project ['song_title, 'qualities[acousticness] AS acousticness#1092, 'qualities[danceability] AS danceability#1093, 'qualities[tempo] AS tempo#1094, 'qualities[energy] AS energy#1095, 'qualities[liveness] AS liveness#1096, 'qualities[instrumentalness] AS instrumentalness#1097, 'qualities[speechiness] AS speechiness#1098, 'qualities[loudness] AS loudness#1099]
+- Project [id#716, song_title#717, artist#718, duration_ms#721, array(key#724, mode#727, target#732) AS audience#837, map(acousticness, acousticness#719, danceability, acousticness#719, energy, energy#722, instrumentalness, instrumentalness#723, liveness, liveness#725, loudness, loudness#726, speechiness, speechiness#728, tempo, tempo#729) AS qualities#838, time_signature#730, valence#731]
   +- Relation [id#716,song_title#717,artist#718,acousticness#719,danceability#720,duration_ms#721,energy#722,instrumentalness#723,key#724,liveness#725,loudness#726,mode#727,speechiness#728,tempo#729,time_signature#73

----------------------------------------
Exception occurred during processing of request from ('127.0.0.1', 60110)
Traceback (most recent call last):
  File "/usr/lib/python3.10/socketserver.py", line 316, in _handle_request_noblock
    self.process_request(request, client_address)
  File "/usr/lib/python3.10/socketserver.py", line 347, in process_request
    self.finish_request(request, client_address)
  File "/usr/lib/python3.10/socketserver.py", line 360, in finish_request
    self.RequestHandlerClass(request, client_address, self)
  File "/usr/lib/python3.10/socketserver.py", line 747, in __init__
    self.handle()
  File "/usr/local/lib/python3.10/dist-packages/pyspark/accumulators.py", line 295, in handle
    poll(accum_updates)
  File "/usr/local/lib/python3.10/dist-packages/pyspark/accumulators.py", line 267, in poll
    if self.rfile in r and func():
  File "/usr/local/lib/python3.10/dist-packages/pyspark/accumulators.py", line 271, in accum_updates
    num_updates = read_int(

## The End

In [ ]:
spark.stop()